In [ ]:
# [장비 준비] 퍼스널 컬러 감별사 육성 아카데미 특훈 장비 챙기기!
# - torch / torchvision: 딥러닝 체육관 & 시각 훈련 교재
# - pandas: 매일매일 모의고사 성적 기록할 종합 생활기록부
# - timm: 세상 온갖 사물을 다 보고 온 '눈썰미 만렙' 천재 유학생(사전학습 비전 모델) 스카우트 센터
# - torchmetrics: 채점관의 4차원 정밀 채점기(정확도, F1, 정밀도, 재현율)
# - wakepy: 밤샘 특훈 중 훈련소 조명(화면) 꺼지지 않게 레드불 먹여주기
# - onnxruntime: 실전 현장(서비스 배포)용 가벼운 실전 압축 패키징 도구
%pip install torch torchvision pandas torchmetrics timm wakepy onnxruntime

In [1]:
import os 
import torch
from torch.nn import CrossEntropyLoss
from torch.optim import AdamW
import pandas as pd
from model import Model                         # 눈썰미 천재 유학생의 두뇌 끝에 '퍼스널 컬러 판정 자격증'을 장착한 특훈 모델
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchmetrics.classification import Accuracy, F1Score, Recall, Precision
from torchmetrics import MetricCollection
from wakepy import keep                          # "공부 중에 졸지 마!" 수면 방지 에너지드링크


/home/an0jin/.local/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 1. 훈련 계획표(하이퍼파라미터) & 특훈 장소 배정

In [2]:
# --- 훈련 강도 및 장소 세팅 ---
BATCH_SIZE = 64                                # 한 번에 모아서 채점할 문제 묶음 (64문제씩 몰아서 풀기)
EPOCHS = 300                                   # 300일간의 스파르타 합숙 훈련 코스
LR = 1e-3                                      # 코치의 피드백 반영 보폭 (학습률: 너무 크면 뇌절하고, 너무 작으면 세월아 네월아)
IMGZ = (384, 384)                              # 훈련생 시야 해상도 (384x384 고화질로 꼼꼼하게 관찰)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # 특훈 훈련소: 슈퍼카 전용 트랙(GPU) or 두 다리로 달리기(CPU)


# 2. 문제집 조련 (어떤 조명·각도에서도 맞히는 매의 눈 훈련)

In [3]:
# --- [연습 모드] 온갖 극한 상황에서도 맞히게 만드는 '매운맛 문제집' 왜곡 파이프라인 ---
train_transform = transforms.Compose([
    transforms.Resize(IMGZ),                                           # 모든 사진 규격 깔끔하게 맞추기
    transforms.RandomHorizontalFlip(),                                 # 거울 보듯 좌우 반전 훈련 (왼쪽 얼굴, 오른쪽 얼굴 편식 금지)
    transforms.RandAugment(num_ops=2, magnitude=9),                    # 무작위 시련 세트 (색감 비틀기, 왜곡 등 랜덤 돌발 상황 대응)
    transforms.ColorJitter(hue=0.015, saturation=0.7, brightness=0.4), # 노란 조명, 백열등, 어두운 카페 조명 테러 대비 (색조는 미세하게)
    transforms.ToTensor(),                                             # 사진 이미지를 컴퓨터가 이해하는 숫자 점수판으로 변환 [0, 255] -> [0.0, 1.0]
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),  # 모든 데이터의 체급을 [-1, 1] 범위로 평등하게 맞추기
    transforms.RandomErasing(p=0.4)                                    # 마스크나 앞머리로 얼굴 일부 가려져도 알아맞히는 궁극의 스파르타
])

# --- [실전 모의고사] 군더더기 없는 표준 시험지 (컨닝·왜곡 금지) ---
val_transform = transforms.Compose([
    transforms.Resize(IMGZ),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# 문제집 캐비닛에서 사진 꺼내오기 (폴더 이름 = 퍼스널 컬러 정답 라벨)
image_datasets = {
    'train': datasets.ImageFolder('./dataset/train', transform=train_transform),
    'val': datasets.ImageFolder('./dataset/val', transform=val_transform)
}

# 문제지 배분기 (연습 때는 문제 순서 외우는 꼼수 방지용 카드 셔플, 시험 때는 정직하게 순서대로)
dataloaders = {
    phase: DataLoader(image_datasets[phase], batch_size=BATCH_SIZE, shuffle=(phase == 'train'))
    for phase in ['train', 'val']
}


# 3. 훈련생(모델), 엄격한 채점관(손실함수), 족집게 코치(옵티마이저) & 졸업장 보관소

In [4]:
# 1. 시각 능력 만렙 유학생(EfficientNetV2)을 특훈 훈련장(GPU)으로 모셔오기
model = Model(image_datasets['train'].classes, 'tf_efficientnetv2_s.in21k_ft_in1k').to(DEVICE)

# 2. 엄격한 채점관: 정답과 헛소리의 괴리감을 측정해 등짝 때리는 채점표 (CrossEntropyLoss)
criterion = CrossEntropyLoss()

# 3. 족집게 코치: 눈썰미(Backbone)는 이미 천재니 건들지 말고, 마지막 '말단 판단소(Classifier)' 뇌세포만 집중 교정
optimizer = AdamW(model.classifier.parameters(), lr=LR)

# 4. 훈련 회차별 일기장 폴더 번호 자동 채번 (run/train, run/train1, run/train2, ...)
folder_index = 0
while os.path.exists(save_path := f"run/train{'' if folder_index == 0 else folder_index}"):
    folder_index += 1
os.makedirs(save_path)
print(f"저장 경로: {save_path}")

# 5. 이번 기수에서 마스터해야 할 퍼스널 컬러 자격증 종목 목록표 저장
with open(f"{save_path}/classes.txt", "w") as file:
    file.write("\n".join(model.classes))


저장 경로: run/train5


# 4. 채점관의 다각도 성적표 기준 (단순 찍기 방지용 4차원 입체 평가)

In [5]:
# 단순히 찍어서 맞힌 것 말고, 4방면 입체 성적표 정의 (정확도, 조화평균 F1, 꼼꼼함 Precision, 빠뜨림 없는 Recall)
base_metrics = MetricCollection({
    'Acc': Accuracy(task='multiclass', num_classes=model.num_classes),
    'F1': F1Score(task='multiclass', num_classes=model.num_classes, average='macro'),
    'Prec': Precision(task='multiclass', num_classes=model.num_classes, average='macro'),
    'Rec': Recall(task='multiclass', num_classes=model.num_classes, average='macro')
})

# 연습용 성적표(train_*)와 모의고사용 성적표(val_*)를 각각 복사해서 훈련장 모니터에 세팅
metrics = {
    phase: base_metrics.clone(prefix=f'{phase}_').to(DEVICE)
    for phase in ['train', 'val']
}


# 5. 300일 지옥의 합숙 루프 (졸지 않게 불 켜두고 실전 특훈)

In [ ]:
best_val_acc = 0.0
history = []
best_val_loss = float('inf')
dump = (torch.randn(1, 3, IMGZ[0], IMGZ[1]),)  # 실전 현장 납품용 신체 치수 측정용 더미 마네킹

# wakepy: 훈련 끝날 때까지 모니터 꺼지거나 졸지 말라고 눈꺼풀 테이프 붙이기
with keep.presenting():
    for epoch in range(EPOCHS):
        epoch_results = {}
        
        for phase in ['train', 'val']:
            # 모드 스위치: 연습 훈련(피드백 수용 모드) vs 실전 모의고사(컨닝 및 수정 불가 평가 모드)
            model.train() if phase == 'train' else model.eval()
            running_loss = 0.0
            metrics[phase].reset()                               # 오늘 시험용 백지 성적표 준비
            
            # 연습 때는 오답 피드백 뇌세포 추적(Grad ON), 모의고사 땐 계산량 아끼게 피드백 끄기(Grad OFF)
            with torch.set_grad_enabled(phase == 'train'):
                for inputs, labels in dataloaders[phase]:
                    inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
                    
                    if phase == 'train':
                        # [1. 칠판 지우기 - zero_grad]
                        # 이전 문제 풀 때 적어둔 힌트(이전 배치의 그래디언트)를 깨끗이 지웁니다.
                        # 지우지 않으면 이전 문제 힌트가 누적되어 엉뚱한 방향으로 다이얼을 돌리게 됩니다.
                        optimizer.zero_grad()
                        
                    # [2. 문제 풀기 (순전파, Forward)]
                    # 이미지를 보고 학생이 "이 사람은 웜톤 70%, 쿨톤 30%!" 하고 답안지를 냅니다.
                    outputs = model(inputs)
                    
                    # [3. 채점하기 (손실, Loss)]
                    # 채점관이 "땡! 정답은 쿨톤이야!" 하면서 정답과 예측의 오차(등짝 스매싱 세기)를 계산합니다.
                    loss = criterion(outputs, labels)
                    
                    if phase == 'train':
                        # [4. 오답 원인 역추적 수사 (역전파, Backpropagation) & 그래디언트(기울기) 계산]
                        # - 역전파: 채점표(결과)에서부터 거꾸로 입력층 방향으로 거슬러 올라가며
                        #          "어떤 뇌세포(가중치)가 이번 오답에 가장 큰 헛발질을 했는가?"를 역추적합니다.
                        # - 그래디언트: "이 다이얼은 시계 방향으로 0.2, 저 다이얼은 반시계로 0.05 돌려야 오답이 줄어든다!"
                        #              라는 최적의 수정 방향과 크기(나침반)를 계산해 각 파라미터에 딱지로 붙여둡니다.
                        loss.backward()
                        
                        # [5. 뇌세포 다이얼 실제로 돌리기 (가중치 갱신, Optimizer Step)]
                        # 딱지에 적힌 그래디언트 방향대로 뇌세포 다이얼을 학습률(LR) 보폭만큼 살짝 돌려서 실력을 1단계 업그레이드합니다.
                        optimizer.step()
                        
                    running_loss += loss.item() * inputs.size(0)
                    metrics[phase].update(outputs, labels)       # 이번 문제 묶음 점수 누적 기록
                    
            phase_loss = running_loss / len(image_datasets[phase])
            phase_metrics = {name: val.item() for name, val in metrics[phase].compute().items()}
            
            epoch_results.update(phase_metrics)
            epoch_results[f'{phase}_Loss'] = phase_loss
            
            metrics_str = " | ".join(f"{name}: {val:.4f}" for name, val in phase_metrics.items())
            print(f"Epoch {epoch+1}/{EPOCHS} [{phase.upper()}] Loss: {phase_loss:.4f} | {metrics_str}")
            
        history.append(epoch_results)
        
        # 오늘 하루 공부 결과 성적표를 생활기록부(CSV)에 실시간 업데이트
        history_df = pd.DataFrame(history)
        history_df.index = history_df.index + 1
        history_df.to_csv(f'{save_path}/result.csv', index_label="epoch")
        
        # 오늘의 훈련생 상태를 가벼운 실전 압축 포맷(ONNX)으로 박제 (last.onnx)
        onnx_model = torch.onnx.export(model, dump, dynamo=True)
        onnx_model.save(f"{save_path}/last.onnx")
        
        # 역대급 모의고사 최고점(리즈 시절) 갱신 시 골든 트로피로 박제 (best.onnx)
        val_acc, val_loss = epoch_results['val_Acc'], epoch_results['val_Loss']
        if (val_acc, -val_loss) > (best_val_acc, -best_val_loss):
            best_val_acc, best_val_loss = val_acc, val_loss
            onnx_model.save(f"{save_path}/best.onnx")
